# OIBSIP Data Analytics – Level 1 – Task 4\n# Sentiment Analysis\n\nClassify text as positive, negative or neutral using TF-IDF and two classifiers.

In [ ]:
import re, pandas as pd, matplotlib.pyplot as plt, seaborn as sns\nfrom sklearn.feature_extraction.text import TfidfVectorizer\nfrom sklearn.model_selection import train_test_split\nfrom sklearn.naive_bayes import MultinomialNB\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.metrics import accuracy_score,precision_recall_fscore_support,classification_report,confusion_matrix\nfrom wordcloud import WordCloud\ndf=pd.read_csv('sentiment_reviews.csv')\nprint(df['Sentiment'].value_counts())\nprint(df.isnull().sum())

## Text preprocessing\nLowercase, punctuation removal, stopword removal and tokenisation are applied before TF-IDF.

In [ ]:
try:
    from nltk.corpus import stopwords
    STOP_WORDS=set(stopwords.words("english"))
except Exception:
    STOP_WORDS={"the","is","a","an","and","or","to","of","in","on","for","with","this","that","it","i","my","was","are"}
def preprocess(text):
    text=re.sub(r"[^a-z\\s]"," ",str(text).lower())
    return " ".join(w for w in text.split() if w not in STOP_WORDS)
df=df.dropna(subset=["Review","Sentiment"]).drop_duplicates("Review").copy()
df["Clean_Review"]=df["Review"].apply(preprocess)
df.head()

## TF-IDF\nTF-IDF converts text into numerical features by giving useful weight to informative words. Bigrams are included too.

In [ ]:
X_train_text,X_test_text,y_train,y_test=train_test_split(df["Clean_Review"],df["Sentiment"],test_size=.20,random_state=42,stratify=df["Sentiment"])
vectorizer=TfidfVectorizer(ngram_range=(1,2))
X_train=vectorizer.fit_transform(X_train_text); X_test=vectorizer.transform(X_test_text)
print(X_train.shape,X_test.shape)

## Train two classifiers: Naive Bayes and Logistic Regression

In [ ]:
models={"Naive Bayes":MultinomialNB(),"Logistic Regression":LogisticRegression(max_iter=1000)}
results=[]; predictions={}
for name,m in models.items():
    m.fit(X_train,y_train); pred=m.predict(X_test); predictions[name]=pred
    p,r,f1,_=precision_recall_fscore_support(y_test,pred,average="weighted",zero_division=0)
    results.append([name,accuracy_score(y_test,pred),p,r,f1])
pd.DataFrame(results,columns=["Model","Accuracy","Precision","Recall","F1-score"]).round(4)

## Confusion matrices and classification reports

In [ ]:
labels=["negative","neutral","positive"]
for name,pred in predictions.items():
    print(name); print(classification_report(y_test,pred,zero_division=0))
    cm=confusion_matrix(y_test,pred,labels=labels)
    sns.heatmap(cm,annot=True,fmt="d",cmap="Blues",xticklabels=labels,yticklabels=labels)
    plt.title(name+" - Confusion Matrix"); plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.show()

## Sentiment distribution

In [ ]:
sns.countplot(data=df,x="Sentiment",order=["negative","neutral","positive"]); plt.title("Sentiment Distribution"); plt.show()

## WordCloud for each sentiment

In [ ]:
for s in ["negative","neutral","positive"]:
    text=" ".join(df.loc[df.Sentiment==s,"Clean_Review"])
    plt.figure(figsize=(10,5)); plt.imshow(WordCloud(width=900,height=450,background_color="white").generate(text)); plt.axis("off"); plt.title("WordCloud - "+s.title()); plt.show()

## Error analysis – 5 misclassified examples

In [ ]:
pred=predictions["Logistic Regression"]
errors=pd.DataFrame({"Review":df.loc[X_test_text.index,"Review"],"Actual":y_test,"Predicted":pred})
errors[errors.Actual!=errors.Predicted].head(5)

## Conclusion\nReport the model with the higher weighted F1-score on this test split. Real-world uses include customer-feedback monitoring, product-review analysis, social-media opinion tracking and support-ticket prioritisation.